# Visual Token Compression for Efficient VLM Inference
### HPML Course Project - LLaVA-1.5-7B + VQAv2

**Pinned versions:**
- `transformers==4.31.0` + `tokenizers==0.13.3`
- `accelerate==0.21.0`
- `torch` — Colab default (2.x OK)

**Key fix:** Flash attention disabled globally for LLaVA compatibility.

**Run order:** Cell 02 installs deps -> **Restart Runtime** -> Cell 03 onward.

| Section | Content |
|---------|--------|
| 0 | Environment Setup |
| 1 | Dataset Preparation |
| 2 | Model Loading + Hook Injection |
| 3 | Compression Strategies |
| 4 | Benchmark Runner |
| 5 | Run All Experiments |
| 6 | Results & Visualization |
| 7 | Fine-grained Analysis |
| 8 | Interactive Demo |

## Section 0 - Environment Setup
Run Cell 02, then **Runtime > Restart Runtime**, then continue from Cell 03.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR  = '/content/drive/MyDrive/HPML_Project'
CACHE_DIR = '/content/drive/MyDrive/HPML_Project/hf_cache'
os.makedirs(SAVE_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

os.environ['HF_DATASETS_CACHE']     = CACHE_DIR
os.environ['HF_HOME']               = CACHE_DIR
os.environ['TRANSFORMERS_CACHE']    = CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = CACHE_DIR
print(f'Cache  -> {CACHE_DIR}')
print(f'Results-> {SAVE_DIR}')

# Core version pins — do NOT change these
# transformers 4.31.0 is the version where LLaVA generate() works correctly
# with torch 2.x + CUDA 12.x on Python 3.12
!pip install -q transformers==4.31.0
!pip install -q tokenizers==0.13.3
!pip install -q accelerate==0.21.0
!pip install -q bitsandbytes
!pip install -q sentencepiece==0.1.99
!pip install -q einops==0.6.1
!pip install -q einops-exts==0.0.4
!pip install -q timm==0.6.13
!pip install -q markdown2 shortuuid
!pip install -q Pillow matplotlib pandas seaborn datasets
!pip install -q --no-deps git+https://github.com/haotian-liu/LLaVA.git

print('\nDone. >>> Restart Runtime now, then run Cell 03 onward. <<<')

In [ ]:
# Run after Restart Runtime — restores env vars and verifies versions
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR  = '/content/drive/MyDrive/HPML_Project'
CACHE_DIR = '/content/drive/MyDrive/HPML_Project/hf_cache'
os.environ['HF_DATASETS_CACHE']     = CACHE_DIR
os.environ['HF_HOME']               = CACHE_DIR
os.environ['TRANSFORMERS_CACHE']    = CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = CACHE_DIR

import torch
# Disable flash attention globally — required for LLaVA on torch 2.x + Python 3.12
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

assert torch.cuda.is_available(), 'No GPU! Switch to A100 in Runtime settings.'
print(f'Torch   : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Flash attention disabled: OK')

import transformers
print(f'Transformers: {transformers.__version__}')
assert transformers.__version__.startswith('4.31'), \
    f'Wrong version: {transformers.__version__}. Re-run Cell 02 and restart.'

## Section 1 - Dataset Preparation
First run: ~10-20 min download to Drive cache. Subsequent runs: instant.

In [ ]:
from datasets import load_dataset
import pandas as pd

print('Loading VQAv2 (1000 samples)...')
raw_ds = load_dataset('lmms-lab/VQAv2', split='validation[:1000]')

samples = []
for item in raw_ds:
    ans_list = item['answers']
    counts = {}
    for a in ans_list:
        a_str = a.strip().lower() if isinstance(a, str) else a['answer'].strip().lower()
        counts[a_str] = counts.get(a_str, 0) + 1
    gt = max(counts, key=counts.get)
    samples.append({
        'question_id': item.get('question_id', 0),
        'question':    item['question'],
        'gt_answer':   gt,
        'answer_type': item.get('answer_type', 'other'),
        'image':       item['image'],
    })

print(f'Loaded {len(samples)} samples')
print(pd.Series([s['answer_type'] for s in samples]).value_counts())
print(f'Example Q: {samples[0]["question"]}')
print(f'Example A: {samples[0]["gt_answer"]}')

## Section 2 - Model Loading + Attention Hook Injection
LLaVA-1.5-7B in float16 (~14 GB VRAM). A100 has 40 GB so no problem.
First run downloads weights to Drive cache (~2 min after that).

In [ ]:
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path
import torch

model_path = 'liuhaotian/llava-v1.5-7b'
model_name = get_model_name_from_path(model_path)

# Load in float16 — no quantization needed, A100 has enough VRAM
# Note: flash attention already disabled globally in Cell 03
tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,
    model_name=model_name,
    load_4bit=False,
    load_8bit=False,
    device_map='auto',
    torch_dtype=torch.float16,
)
model.eval()
print('Model loaded!')
print(f'LLM layers: {model.model.config.num_hidden_layers}')
print(f'VRAM used : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
import torch

_captured_attn = {}
_hook_handles  = []
N_VISUAL = 576

def _make_attn_hook(layer_idx):
    def hook(module, input, output):
        if isinstance(output, tuple) and len(output) > 1 and output[1] is not None:
            _captured_attn[layer_idx] = output[1].detach().cpu()
    return hook

def register_hooks(layers=(2, 4, 8, 16)):
    global _hook_handles
    for h in _hook_handles:
        h.remove()
    _hook_handles = []
    for idx, layer in enumerate(model.model.layers):
        if idx in layers:
            h = layer.self_attn.register_forward_hook(_make_attn_hook(idx))
            _hook_handles.append(h)
    print(f'Hooks registered on layers: {list(layers)}')

register_hooks()

## Section 3 - Compression Strategies

Interface: `fn(hidden_states, attn_weights, keep_ratio) -> (compressed_hs, kept_idx)`

Only visual tokens `[:N_VISUAL]` are modified. Text tokens are never touched.

In [ ]:
import torch
import torch.nn.functional as F
import math

# Strategy 1: Uniform Pruning
# Drop tokens at fixed intervals. No attention signal. Pure baseline.
def uniform_prune(hidden_states, attn_weights, keep_ratio):
    k   = max(1, int(N_VISUAL * keep_ratio))
    idx = torch.linspace(0, N_VISUAL - 1, k).long().to(hidden_states.device)
    vis = hidden_states[:, :N_VISUAL]
    txt = hidden_states[:, N_VISUAL:]
    return torch.cat([vis[:, idx, :], txt], dim=1), idx

# Strategy 2: Attention-guided Pruning  <-- CORE METHOD
# Score visual tokens by mean attention from text tokens. Keep top-k.
# Extends FastV (Chen et al., ECCV 2024).
def attention_prune(hidden_states, attn_weights, keep_ratio):
    k              = max(1, int(N_VISUAL * keep_ratio))
    attn_weights   = attn_weights.to(hidden_states.device)
    text_to_visual = attn_weights[:, :, N_VISUAL:, :N_VISUAL]
    importance     = text_to_visual.mean(dim=(1, 2))
    topk_idx       = importance[0].topk(k).indices.sort().values
    vis            = hidden_states[:, :N_VISUAL]
    txt            = hidden_states[:, N_VISUAL:]
    return torch.cat([vis[:, topk_idx, :], txt], dim=1), topk_idx

# Strategy 3: Token Merging (ToMe-style)
# Merge similar visual tokens via cosine similarity bipartite matching.
# Based on Bolya et al. (ICLR 2023).
def token_merge(hidden_states, attn_weights, keep_ratio):
    k       = max(1, int(N_VISUAL * keep_ratio))
    n_merge = N_VISUAL - k
    vis     = hidden_states[:, :N_VISUAL]
    txt     = hidden_states[:, N_VISUAL:]
    merged  = vis[0].clone()
    v_norm  = F.normalize(merged, dim=-1)
    sim     = torch.mm(v_norm, v_norm.t())
    sim.fill_diagonal_(-1)
    mask    = torch.ones(N_VISUAL, dtype=torch.bool, device=hidden_states.device)
    for _ in range(n_merge):
        valid = sim.clone()
        valid[~mask, :] = -1
        valid[:, ~mask] = -1
        flat  = valid.argmax()
        i, j  = flat // N_VISUAL, flat % N_VISUAL
        if not mask[i] or not mask[j] or i == j:
            break
        merged[i] = (merged[i] + merged[j]) / 2
        mask[j]   = False
        sim[j, :] = -1
        sim[:, j] = -1
    kept     = mask.nonzero(as_tuple=True)[0]
    merged_h = merged[kept].unsqueeze(0)
    return torch.cat([merged_h, txt], dim=1), kept

# Strategy 4: Spatial Pooling
# Reshape 576 tokens as 24x24 grid and average pool to target size.
def spatial_pool(hidden_states, attn_weights, keep_ratio):
    k       = max(1, int(N_VISUAL * keep_ratio))
    grid    = int(math.sqrt(N_VISUAL))  # 24
    tgt     = max(1, int(math.sqrt(k)))
    vis     = hidden_states[:, :N_VISUAL]
    txt     = hidden_states[:, N_VISUAL:]
    B, _, D = vis.shape
    g       = vis.permute(0, 2, 1).reshape(B, D, grid, grid)
    pooled  = F.adaptive_avg_pool2d(g, (tgt, tgt))
    pooled  = pooled.reshape(B, D, -1).permute(0, 2, 1)
    kept    = torch.arange(tgt * tgt, device=hidden_states.device)
    return torch.cat([pooled, txt], dim=1), kept

STRATEGIES = {
    'baseline':  None,
    'uniform':   uniform_prune,
    'attention': attention_prune,
    'merge':     token_merge,
    'pool':      spatial_pool,
}
print('Strategies:', list(STRATEGIES.keys()))

## Section 4 - Benchmark Runner
Uses `model.generate()` with flash attention disabled (set globally in Cell 03).
Compression hook intercepts hidden states after pruning layer k.

In [ ]:
import time
import numpy as np
from llava.mm_utils import process_images, tokenizer_image_token
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from llava.conversation import conv_templates


def vqa_score(pred, gt):
    return 1.0 if pred.strip().lower() == gt.strip().lower() else 0.0


def run_inference_single(sample, strategy_fn, keep_ratio, prune_layer=4):
    image    = sample['image'].convert('RGB')
    question = sample['question']

    image_tensor = process_images([image], image_processor, model.config)
    image_tensor = image_tensor.to(model.device, dtype=torch.float16)

    conv = conv_templates['vicuna_v1'].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + '\n' + question)
    conv.append_message(conv.roles[1], None)

    input_ids = tokenizer_image_token(
        conv.get_prompt(), tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt'
    ).unsqueeze(0).to(model.device)

    attention_mask = torch.ones_like(input_ids)

    _captured_attn.clear()
    tokens_used  = [N_VISUAL]
    patch_handle = [None]

    if strategy_fn is not None:
        def compress_hook(module, input, output):
            if prune_layer not in _captured_attn:
                return output
            hs     = output[0]
            attn_w = _captured_attn[prune_layer]
            n_text = input_ids.shape[1] - N_VISUAL
            if hs.shape[1] > int(N_VISUAL * keep_ratio) + n_text:
                compressed, kept = strategy_fn(hs, attn_w, keep_ratio)
                tokens_used[0]   = kept.shape[0]
                return (compressed,) + output[1:]
            return output
        attach_at       = min(prune_layer + 1, len(model.model.layers) - 1)
        patch_handle[0] = model.model.layers[attach_at].register_forward_hook(compress_hook)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.perf_counter()

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=image_tensor,
            attention_mask=attention_mask,
            do_sample=False,
            max_new_tokens=32,
            use_cache=True,
        )

    torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - t0) * 1000
    peak_vram  = torch.cuda.max_memory_allocated() / 1e9

    if patch_handle[0] is not None:
        patch_handle[0].remove()

    # Decode: output_ids may contain only new tokens (len <= input_ids)
    if output_ids.shape[1] > input_ids.shape[1]:
        pred = tokenizer.decode(
            output_ids[0, input_ids.shape[1]:], skip_special_tokens=True
        ).strip()
    else:
        pred = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

    return pred, latency_ms, peak_vram, tokens_used[0]


# Quick sanity check
print('Testing single inference...')
pred, lat, vram, n_tok = run_inference_single(samples[0], None, 1.0)
print(f'Q  : {samples[0]["question"]}')
print(f'A  : {pred}')
print(f'GT : {samples[0]["gt_answer"]}')
print(f'Latency: {lat:.0f}ms | VRAM: {vram:.2f}GB | Tokens: {n_tok}')

In [ ]:
def run_experiment(strategy_name, keep_ratio, samples, prune_layer=4, verbose=True):
    strategy_fn = STRATEGIES[strategy_name]
    if strategy_name == 'baseline':
        keep_ratio = 1.0

    scores, latencies, vrams, token_counts = [], [], [], []
    by_type = {'yes/no': [], 'number': [], 'other': []}

    for i, sample in enumerate(samples):
        pred, lat, vram, n_tok = run_inference_single(
            sample, strategy_fn, keep_ratio, prune_layer
        )
        acc = vqa_score(pred, sample['gt_answer'])
        scores.append(acc)
        latencies.append(lat)
        vrams.append(vram)
        token_counts.append(n_tok)
        atype = sample.get('answer_type', 'other')
        if atype in by_type:
            by_type[atype].append(acc)
        if verbose and (i + 1) % 100 == 0:
            print(f'  [{i+1}/{len(samples)}]  '
                  f'acc={np.mean(scores):.3f}  '
                  f'lat={np.mean(latencies):.0f}ms  '
                  f'vram={np.mean(vrams):.2f}GB')

    result = {
        'strategy':     strategy_name,
        'keep_ratio':   keep_ratio,
        'prune_layer':  prune_layer,
        'accuracy':     np.mean(scores),
        'latency_ms':   np.mean(latencies),
        'peak_vram_gb': np.mean(vrams),
        'tokens_used':  np.mean(token_counts),
        'acc_yesno':    np.mean(by_type['yes/no'])  if by_type['yes/no']  else None,
        'acc_number':   np.mean(by_type['number'])  if by_type['number']  else None,
        'acc_other':    np.mean(by_type['other'])   if by_type['other']   else None,
        'n_samples':    len(samples),
    }
    if verbose:
        print(f'\n[DONE] {strategy_name} @ {keep_ratio:.0%} | '
              f'acc={result["accuracy"]:.3f} | '
              f'lat={result["latency_ms"]:.0f}ms | '
              f'vram={result["peak_vram_gb"]:.2f}GB | '
              f'tokens={result["tokens_used"]:.0f}')
    return result

print('run_experiment() ready.')

## Section 5 - Run All Experiments

**Estimated time on A100: ~2-2.5 hours**

Results auto-save to Drive after each experiment.
If Colab disconnects: re-run Cells 03-13, then re-run this section. Done experiments are skipped.

In [ ]:
import pandas as pd
import os

RESULTS_PATH = os.path.join(SAVE_DIR, 'results.csv')

if os.path.exists(RESULTS_PATH):
    results_df = pd.read_csv(RESULTS_PATH)
    results    = results_df.to_dict('records')
    print(f'Resumed: {len(results)} experiments loaded from Drive.')
else:
    results = []
    print('Starting fresh.')

def already_done(strategy, keep_ratio, prune_layer=4):
    for r in results:
        if (r['strategy'] == strategy and
            abs(r['keep_ratio'] - keep_ratio) < 0.01 and
            r['prune_layer'] == prune_layer):
            return True
    return False

In [ ]:
# Main Experiment Matrix
# Exp-0 : Baseline                 (no compression)
# Exp-1 : Uniform Pruning          @ 75% / 50% / 25%
# Exp-2 : Attention-guided Pruning @ 75% / 50% / 25%  <-- CORE
# Exp-3 : Token Merging            @ 75% / 50% / 25%
# Exp-4 : Spatial Pooling          @ 75% / 50% / 25%

RATIOS = [0.75, 0.50, 0.25]

for strategy in ['baseline', 'uniform', 'attention', 'merge', 'pool']:
    ratios = [1.0] if strategy == 'baseline' else RATIOS
    for ratio in ratios:
        if already_done(strategy, ratio):
            print(f'[skip] {strategy} @ {ratio:.0%}')
            continue
        print(f'\n[RUN] {strategy} @ {ratio:.0%} ...')
        r = run_experiment(strategy, ratio, samples, prune_layer=4)
        results.append(r)
        pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)
        print(f'  -> saved.')

print('\nMain experiments complete!')

In [ ]:
# Exp-5: Ablation - Pruning Layer Position
# attention @ 50%, vary k in {2, 8, 16}

for k in [2, 8, 16]:
    if already_done('attention', 0.50, prune_layer=k):
        print(f'[skip] attention @ 50% layer={k}')
        continue
    print(f'\n[RUN] attention @ 50%, layer={k}...')
    r = run_experiment('attention', 0.50, samples, prune_layer=k)
    r['prune_layer'] = k
    results.append(r)
    pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

results_df = pd.DataFrame(results)
print('\nAll experiments complete!')
print(results_df[['strategy','keep_ratio','prune_layer',
                   'accuracy','latency_ms','peak_vram_gb','tokens_used']].to_string())

## Section 6 - Results & Visualization
4 figures + full summary table.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

results_df = pd.read_csv(RESULTS_PATH)
main_df    = results_df[results_df['prune_layer'] == 4].copy()

COLORS  = {'baseline':'#2d3436','uniform':'#e17055',
           'attention':'#0984e3','merge':'#00b894','pool':'#6c5ce7'}
MARKERS = {'baseline':'D','uniform':'s','attention':'o','merge':'^','pool':'P'}
LABELS  = {
    'baseline' : 'Baseline (no compression)',
    'uniform'  : 'Uniform Pruning',
    'attention': 'Attention-guided Pruning',
    'merge'    : 'Token Merging (ToMe)',
    'pool'     : 'Spatial Pooling',
}

sns.set_theme(style='whitegrid', font_scale=1.1)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LLaVA-1.5-7B - Visual Token Compression Results',
             fontsize=15, fontweight='bold')

# Fig 1: Pareto Curve
ax = axes[0, 0]
for strat, grp in main_df.groupby('strategy'):
    grp = grp.sort_values('keep_ratio', ascending=False)
    ax.plot(grp['latency_ms'], grp['accuracy'],
            color=COLORS[strat], marker=MARKERS[strat],
            linewidth=2, markersize=8, label=LABELS[strat])
    if strat != 'baseline':
        for _, row in grp.iterrows():
            ax.annotate(f"{row['keep_ratio']:.0%}",
                        (row['latency_ms'], row['accuracy']),
                        textcoords='offset points', xytext=(5, 3), fontsize=7.5)
ax.set_xlabel('Avg Latency (ms/sample)')
ax.set_ylabel('VQA Accuracy')
ax.set_title('Accuracy vs Latency (Pareto Curve)')
ax.legend(fontsize=8)

# Fig 2: VRAM
ax = axes[0, 1]
for strat, grp in main_df.groupby('strategy'):
    grp = grp.sort_values('keep_ratio')
    ax.plot(grp['keep_ratio']*100, grp['peak_vram_gb'],
            color=COLORS[strat], marker=MARKERS[strat],
            linewidth=2, markersize=8, label=LABELS[strat])
ax.set_xlabel('Token Retention (%)')
ax.set_ylabel('Peak VRAM (GB)')
ax.set_title('Peak GPU Memory vs Retention Ratio')
ax.invert_xaxis()
ax.legend(fontsize=8)

# Fig 3: Token Budget vs Accuracy
ax = axes[1, 0]
for strat, grp in main_df.groupby('strategy'):
    ax.scatter(grp['tokens_used'], grp['accuracy'],
               color=COLORS[strat], marker=MARKERS[strat],
               s=100, label=LABELS[strat], zorder=3)
ax.set_xlabel('Visual Tokens Used')
ax.set_ylabel('VQA Accuracy')
ax.set_title('Token Budget vs Accuracy')
ax.legend(fontsize=8)

# Fig 4: Ablation
ax  = axes[1, 1]
abl = results_df[
    (results_df['strategy'] == 'attention') &
    (results_df['keep_ratio'].between(0.49, 0.51))
].sort_values('prune_layer')
ax.bar(abl['prune_layer'].astype(str), abl['accuracy'],
       color='#0984e3', edgecolor='white')
ax2 = ax.twinx()
ax2.plot(abl['prune_layer'].astype(str), abl['latency_ms'],
         color='#e17055', marker='o', linewidth=2, markersize=8)
ax.set_xlabel('Pruning Layer k')
ax.set_ylabel('VQA Accuracy', color='#0984e3')
ax2.set_ylabel('Latency (ms)', color='#e17055')
ax.set_title('Ablation: Pruning Layer Position (50% retention)')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'results_main.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results_main.png')

In [ ]:
base_acc = main_df[main_df['strategy']=='baseline']['accuracy'].values[0]
base_lat = main_df[main_df['strategy']=='baseline']['latency_ms'].values[0]
summary  = main_df.copy()
summary['acc_drop_%']     = ((base_acc - summary['accuracy']) / base_acc * 100).round(2)
summary['latency_gain_%'] = ((base_lat - summary['latency_ms']) / base_lat * 100).round(2)
cols = ['strategy','keep_ratio','accuracy','acc_drop_%',
        'latency_ms','latency_gain_%','peak_vram_gb','tokens_used']
print('='*95)
print('FULL RESULTS TABLE')
print('='*95)
print(summary[cols].sort_values(['strategy','keep_ratio']).to_string(index=False))

## Section 7 - Fine-grained Analysis by Question Type

Breakdown: **Yes/No** | **Number** | **Open-ended** at 50% retention.

Hypothesis: number/counting questions are more sensitive to token compression
because they require precise spatial reasoning over the full image.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

half_df    = main_df[main_df['keep_ratio'].between(0.49, 0.51)].copy()
all_strats = list(half_df['strategy'].unique())
q_types    = ['acc_yesno','acc_number','acc_other']
q_labels   = ['Yes/No','Number','Open-ended']
x = np.arange(len(q_labels))
w = 0.15

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fine-grained Analysis: Accuracy by Question Type (50% retention)',
             fontsize=13, fontweight='bold')

ax = axes[0]
for i, strat in enumerate(all_strats):
    row  = half_df[half_df['strategy']==strat].iloc[0]
    vals = [row.get(qt) or 0 for qt in q_types]
    ax.bar(x+i*w, vals, w, label=LABELS[strat], color=COLORS[strat], edgecolor='white')
ax.set_xticks(x + w*(len(all_strats)-1)/2)
ax.set_xticklabels(q_labels)
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy per Question Type @ 50%')
ax.legend(fontsize=7.5)
ax.set_ylim(0, 1.05)

ax       = axes[1]
base_row = main_df[main_df['strategy']=='baseline'].iloc[0]
bv       = {qt: base_row.get(qt) or 0 for qt in q_types}
non_base = [s for s in all_strats if s != 'baseline']
for i, strat in enumerate(non_base):
    row   = half_df[half_df['strategy']==strat].iloc[0]
    drops = [((bv[qt]-(row.get(qt) or 0))/max(bv[qt],1e-6))*100 for qt in q_types]
    ax.bar(x+i*w, drops, w, label=LABELS[strat], color=COLORS[strat], edgecolor='white')
ax.set_xticks(x + w*1.5)
ax.set_xticklabels(q_labels)
ax.set_ylabel('Accuracy Drop vs Baseline (%)')
ax.set_title('Sensitivity by Question Type @ 50%')
ax.legend(fontsize=7.5)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'results_finegrained.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results_finegrained.png')

In [ ]:
print('KEY FINDINGS - Question Type Sensitivity @ 50% retention')
print('='*65)
base_row = main_df[main_df['strategy']=='baseline'].iloc[0]
for strat in ['uniform','attention','merge','pool']:
    rows = main_df[(main_df['strategy']==strat) &
                   (main_df['keep_ratio'].between(0.49,0.51))]
    if rows.empty: continue
    row = rows.iloc[0]
    print(f'\n{LABELS[strat]}:')
    for qt, ql in zip(q_types, q_labels):
        bv   = base_row.get(qt) or 0
        sv   = row.get(qt) or 0
        drop = (bv-sv)/max(bv,1e-6)*100
        print(f'  {ql:12s}: {sv:.3f}  (drop: {drop:+.1f}%)')

## Section 8 - Interactive Demo
Upload image, ask question, pick strategy and retention ratio.
Shows model answer + attention heatmap of retained vs pruned tokens.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import io

upload_btn   = widgets.FileUpload(accept='image/*', multiple=False)
question_box = widgets.Text(value='What is in this image?',
                             description='Question:',
                             layout=widgets.Layout(width='500px'))
strategy_dd  = widgets.Dropdown(options=list(STRATEGIES.keys()),
                                 value='attention', description='Strategy:')
ratio_slider = widgets.FloatSlider(value=0.5, min=0.25, max=1.0, step=0.25,
                                    description='Keep %:', readout_format='.0%')
run_btn = widgets.Button(description='Run Inference', button_style='primary')
out     = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        if not upload_btn.value:
            print('Please upload an image first.')
            return
        img_bytes  = list(upload_btn.value.values())[0]['content']
        image      = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        sample     = {'image':image,'question':question_box.value,
                      'gt_answer':'','answer_type':'other'}
        strat_name = strategy_dd.value
        ratio      = ratio_slider.value
        fn         = STRATEGIES[strat_name]
        print(f'Running {strat_name} @ {ratio:.0%}...')
        pred, lat, vram, n_tok = run_inference_single(sample, fn, ratio)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(image)
        axes[0].set_title('Input Image')
        axes[0].axis('off')

        if 4 in _captured_attn and strat_name != 'baseline':
            attn_w = _captured_attn[4].to('cpu')
            tov    = attn_w[0, :, N_VISUAL:, :N_VISUAL].mean(dim=(0,1))
            tov    = tov.float().numpy()
            tov    = (tov-tov.min())/(tov.max()-tov.min()+1e-8)
            hm     = tov.reshape(24, 24)
            hm_up  = np.array(Image.fromarray(
                (hm*255).astype(np.uint8)).resize(image.size, Image.BILINEAR)
            )/255.0
            axes[1].imshow(image, alpha=0.4)
            axes[1].imshow(hm_up, cmap='RdYlBu_r', alpha=0.6, vmin=0, vmax=1)
            axes[1].set_title('Attention Heatmap\n(warm=retained, cool=pruned)')
        else:
            axes[1].imshow(image)
            axes[1].set_title('Baseline - no compression')
        axes[1].axis('off')

        plt.suptitle(
            f'Strategy: {LABELS[strat_name]}  |  Retention: {ratio:.0%}\n'
            f'Answer: "{pred}"  |  Latency: {lat:.0f}ms  |  '
            f'VRAM: {vram:.2f}GB  |  Tokens: {n_tok}',
            fontsize=11)
        plt.tight_layout()
        plt.show()

run_btn.on_click(on_run)
display(widgets.VBox([
    widgets.HTML('<h3>HPML Demo - Visual Token Compression</h3>'),
    upload_btn, question_box,
    widgets.HBox([strategy_dd, ratio_slider]),
    run_btn, out
]))